In [36]:
import asyncio
import hashlib
import inspect
import json
import os
import time
import uuid
from collections import defaultdict
from collections.abc import Callable
from dataclasses import asdict, dataclass, field
from enum import Enum
from typing import Any

from dotenv import load_dotenv
from google.genai import Client
from pydantic import BaseModel, Field, ValidationError

In [ ]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1: Core Agent Loop

In [12]:
def get_stock_price(ticker: str) -> float:
    prices = {
        "AAPL": 235.50,
        "GOOGL": 188.00,
        "MSFT": 430.00,
        "AMZN": 195.00,
    }
    symbol = ticker.upper()
    if symbol not in prices:
        raise ValueError(f"Unknown stock ticker: {ticker}")
    return prices[symbol]


def convert_currency(amount: float, from_currency: str, to_currency: str) -> float:
    rates_to_usd = {
        "USD": 1.0,
        "EUR": 1.08,
        "GBP": 1.30,
        "INR": 0.12,
    }
    src = from_currency.upper()
    dst = to_currency.upper()
    if src not in rates_to_usd or dst not in rates_to_usd:
        raise ValueError(f"Unsupported currency: {src} or {dst}")
    return round((amount * rates_to_usd[src]) / rates_to_usd[dst], 2)


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "name": "get_stock_price",
        "description": "Retrieve current stock price for a given ticker symbol in USD.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker symbol, e.g. AAPL"},
            },
            "required": ["ticker"],
        },
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Amount to convert"},
                "from_currency": {"type": "string", "description": "Source currency, e.g. USD"},
                "to_currency": {"type": "string", "description": "Target currency, e.g. EUR"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
]

TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
}

In [13]:
async def run_agent(
    goal: str,
    client: Any,
    model: str,
    max_steps: int = 5,
) -> str:
    """Core agent loop: observe -> LLM -> execute tools -> append results."""
    # 1. Observe: Initialize agent loop with the user goal

    prompt = f"""
    You are support assistant and you task is help user with available tool.
    If user ask outside of available tools say no clearly.
    User questions :- 
    {goal}
    """

    interaction = client.interactions.create(
        model=model,
        input=prompt,
        tools=TOOL_DEFINITIONS,
    )

    for _ in range(max_steps):
        # 2. Extract tool calls requested by the LLM
        tool_calls = [step for step in (interaction.steps or []) if step.type == "function_call"]

        # Final-answer exit condition: no tool calls means the model produced final answer
        if not tool_calls:
            return interaction.output_text or ""

        # 3. Execute tools
        tool_results = []
        for call in tool_calls:
            result = TOOL_REGISTRY[call.name](**(call.arguments or {}))
            tool_results.append(
                {
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [{"type": "text", "text": json.dumps({"result": result})}],
                }
            )

        # 4. Append results & observe next LLM output
        interaction = client.interactions.create(
            model=model,
            previous_interaction_id=interaction.id,
            input=tool_results,
            tools=TOOL_DEFINITIONS,
        )

    raise RuntimeError("Agent exceeded max steps without reaching a final answer.")

In [ ]:
result = await run_agent(
    goal="Get the stock price of AAPL. and tell me how much in indian rupees",
    client=google_client,
    model=MODEL,
)

print("\nFINAL ANSWER:")
print(result)


FINAL ANSWER:
The current stock price of AAPL is $235.50, which is approximately ₹1,962.50 in Indian Rupees.


### Task 2

In [18]:
class RunState(BaseModel):
    steps: int = 0
    tokens: int = 0
    start_time: float = Field(default_factory=time.time)


class AgentLimits(BaseModel):
    max_steps: int = 5
    token_budget: int = 1000
    deadline_seconds: float = 10.0

    def check(self, run_state: Any) -> str | None:
        """Check operational limits (deadline, steps, tokens).

        Returns:
            A stop reason string ('deadline_exceeded', 'max_steps_exceeded',
            'token_budget_exceeded') or None if execution is within limits.
        """
        # 1. Wall-clock deadline check
        start_time = getattr(run_state, "start_time", None)
        if start_time is None and isinstance(run_state, dict):
            start_time = run_state.get("start_time")
        if start_time is not None and (time.time() - start_time) >= self.deadline_seconds:
            return "deadline_exceeded"

        # 2. Step cap check
        steps = getattr(run_state, "steps", None)
        if steps is None and isinstance(run_state, dict):
            steps = run_state.get("steps")
        if steps is not None and steps >= self.max_steps:
            return "max_steps_exceeded"

        # 3. Token budget check
        tokens = getattr(run_state, "tokens", None)
        if tokens is None and isinstance(run_state, dict):
            tokens = run_state.get("tokens")
        if tokens is not None and tokens >= self.token_budget:
            return "token_budget_exceeded"

        return None

In [19]:
limits = AgentLimits(max_steps=5, token_budget=1000, deadline_seconds=2.0)

# 1. Under limits -> returns None
state_ok = RunState(steps=2, tokens=350)
print("Under limits check:", limits.check(state_ok))  # None

# 2. Step limit breached
state_steps = RunState(steps=5, tokens=350)
print("Steps breached check:", limits.check(state_steps))  # max_steps_exceeded

# 3. Token budget breached
state_tokens = RunState(steps=2, tokens=1200)
print("Tokens breached check:", limits.check(state_tokens))  # token_budget_exceeded

# 4. Deadline breached
state_deadline = RunState(steps=1, tokens=50, start_time=time.time() - 3.0)
print("Deadline breached check:", limits.check(state_deadline))  # deadline_exceeded

Under limits check: None
Steps breached check: max_steps_exceeded
Tokens breached check: token_budget_exceeded
Deadline breached check: deadline_exceeded


### Task 3

In [20]:
def normalize_value(val: Any) -> Any:
    """Recursively normalize values: strip/lower strings, round floats, sort dict keys."""
    if isinstance(val, str):
        return val.strip().lower()
    if isinstance(val, float):
        return round(val, 6)
    if isinstance(val, dict):
        return {k: normalize_value(v) for k, v in sorted(val.items())}
    if isinstance(val, (list, tuple)):
        return [normalize_value(x) for x in val]
    return val


def canonicalize_tool_call(name: str, arguments: dict[str, Any] | None) -> str:
    """Canonicalize tool call with sorted keys and normalized argument values."""
    normalized_args = normalize_value(arguments or {})
    canonical_json = json.dumps(normalized_args, sort_keys=True)
    return hashlib.sha256(f"{name.strip().lower()}:{canonical_json}".encode()).hexdigest()


class LoopDetector:
    """Tracks canonical tool calls and flags any call repeated N times."""

    def __init__(self, max_repeats: int = 3) -> None:
        self.max_repeats = max_repeats
        self.call_counts: dict[str, int] = defaultdict(int)

    def check_and_record(self, name: str, arguments: dict[str, Any] | None) -> bool:
        """Record the tool call; returns True if repeated >= max_repeats (loop detected)."""
        call_hash = canonicalize_tool_call(name, arguments)
        self.call_counts[call_hash] += 1
        return self.call_counts[call_hash] >= self.max_repeats

In [25]:
detector = LoopDetector(max_repeats=3)
# 1. Test canonicalization resilience (differing key order, whitespace, casing produce identical hash)
h1 = canonicalize_tool_call(
    "convert_currency", {"to_currency": " EUR ", "amount": 100.0, "from_currency": "usd"}
)
h2 = canonicalize_tool_call(
    "CONVERT_CURRENCY", {"amount": 100.0, "from_currency": "USD", "to_currency": "eur"}
)
print("Hashes match across different key orders & casing:", h1 == h2)  # True
# 2. Test loop detection flagging
print(
    "Call 1 flagged:", detector.check_and_record("get_stock_price", {"ticker": "AAPL"})
)  # False (1/3)
print(
    "Call 2 flagged:", detector.check_and_record("get_stock_price", {"ticker": " aapl "})
)  # False (2/3)
print(
    "Call 3 flagged:", detector.check_and_record("get_stock_price", {"ticker": "AAPL"})
)  # True (3/3 -> Flagged!)

Hashes match across different key orders & casing: True
Call 1 flagged: False
Call 2 flagged: False
Call 3 flagged: True


# Task 4

In [28]:
@dataclass
class ToolSpec:
    name: str
    fn: Any
    args_schema: type[BaseModel]
    timeout_seconds: float = 5.0


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, ToolSpec] = {}

    def register(
        self,
        name: str,
        fn: Any,
        args_schema: type[BaseModel],
        timeout_seconds: float = 5.0,
    ) -> None:
        self._tools[name] = ToolSpec(
            name=name,
            fn=fn,
            args_schema=args_schema,
            timeout_seconds=timeout_seconds,
        )

    async def execute(
        self,
        name: str,
        arguments: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        """Execute tool with Pydantic validation, per-tool timeout, and structured error handling."""
        arguments = arguments or {}

        # 1. Unknown tool check
        if name not in self._tools:
            return {
                "ok": False,
                "error_type": "ToolNotFoundError",
                "error": f"Tool '{name}' is not registered.",
            }

        tool = self._tools[name]

        # 2. Pydantic argument validation
        try:
            validated = tool.args_schema.model_validate(arguments)
            kwargs = validated.model_dump()
        except ValidationError as exc:
            return {
                "ok": False,
                "error_type": "ValidationError",
                "error": str(exc),
            }

        # 3. Execution with per-tool timeout and structured error handling
        try:
            if inspect.iscoroutinefunction(tool.fn):
                coro = tool.fn(**kwargs)
            else:
                coro = asyncio.to_thread(tool.fn, **kwargs)

            result = await asyncio.wait_for(coro, timeout=tool.timeout_seconds)
            return {"ok": True, "result": result}
        except TimeoutError:
            return {
                "ok": False,
                "error_type": "TimeoutError",
                "error": f"Tool '{name}' timed out after {tool.timeout_seconds}s.",
            }
        except Exception as exc:  # noqa: BLE001
            return {
                "ok": False,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

### Task 5

In [31]:
def truncate_tool_output(content: object, max_length: int = 1000) -> str:
    """Cap tool output length and append a clear truncation notice if exceeded."""
    text = content if isinstance(content, str) else str(content)
    if len(text) <= max_length:
        return text

    overflow = len(text) - max_length
    return f"{text[:max_length]}\n[truncated: {overflow} more characters]"

In [32]:
sample_output = "Server log entry: " + ("OK: Healthy status. " * 20)
print("Original length:", len(sample_output))

print("\nTruncated Output:")
print(truncate_tool_output(sample_output, max_length=100))

Original length: 418

Truncated Output:
Server log entry: OK: Healthy status. OK: Healthy status. OK: Healthy status. OK: Healthy status. OK
[truncated: 318 more characters]


### Task 6

In [34]:
class RiskLevel(str, Enum):
    READ_ONLY = "read_only"
    REVERSIBLE = "reversible"
    IRREVERSIBLE = "irreversible"


@dataclass
class PausedState:
    run_id: str
    tool_name: str
    arguments: dict[str, Any]
    history: list[dict[str, Any]] = field(default_factory=list)
    status: str = "paused_for_approval"


class ApprovalGate:
    """Wrapper that pauses on irreversible tools, persists state, and resumes on decision."""

    def __init__(
        self,
        tool_registry: dict[str, Callable[..., Any]],
        tool_risks: dict[str, RiskLevel],
    ) -> None:
        self.tool_registry = tool_registry
        self.tool_risks = tool_risks
        self.store: dict[str, PausedState] = {}

    def execute_or_pause(
        self,
        tool_name: str,
        arguments: dict[str, Any],
        history: list[dict[str, Any]],
        run_id: str | None = None,
    ) -> dict[str, Any]:
        """Execute safe tools immediately; pause and persist state on irreversible tools."""
        risk = self.tool_risks.get(tool_name, RiskLevel.READ_ONLY)

        # 1. Pause on irreversible tools and persist state
        if risk == RiskLevel.IRREVERSIBLE:
            active_run_id = run_id or str(uuid.uuid4())
            state = PausedState(
                run_id=active_run_id,
                tool_name=tool_name,
                arguments=arguments,
                history=history,
            )
            self.store[active_run_id] = state
            return {
                "status": "paused_for_approval",
                "run_id": active_run_id,
                "tool_name": tool_name,
                "arguments": arguments,
                "message": f"Paused: '{tool_name}' is irreversible and requires human approval.",
            }

        # 2. Reversible / Read-only tools execute immediately
        tool_fn = self.tool_registry[tool_name]
        result = tool_fn(**arguments)
        return {"status": "executed", "ok": True, "result": result}

    def resume(self, run_id: str, decision: str) -> dict[str, Any]:
        """Resume a paused state with an 'approve' or 'reject' decision."""
        if run_id not in self.store:
            raise KeyError(f"Run ID '{run_id}' not found in approval store.")

        state = self.store.pop(run_id)

        if decision.lower() in ("approve", "approved", "yes"):
            tool_fn = self.tool_registry[state.tool_name]
            try:
                result = tool_fn(**state.arguments)
                payload = {"ok": True, "result": result}
            except Exception as exc:  # noqa: BLE001
                payload = {"ok": False, "error": str(exc)}
            status = "approved_and_executed"
        else:
            payload = {
                "ok": False,
                "error": f"Tool '{state.tool_name}' was rejected by human approver.",
            }
            status = "rejected"

        state.history.append({"tool": state.tool_name, "payload": payload})
        return {
            "status": status,
            "decision": decision,
            "tool_name": state.tool_name,
            "payload": payload,
            "updated_history": state.history,
        }

In [35]:
tools = {
    "lookup_account": lambda id: f"CustomerAccount(id={id}, balance=1250.00)",
    "execute_refund": lambda id, amount: f"Refunded ${amount} to account {id}",
}

risks = {
    "lookup_account": RiskLevel.READ_ONLY,
    "execute_refund": RiskLevel.IRREVERSIBLE,
}

gate = ApprovalGate(tool_registry=tools, tool_risks=risks)

# 1. Read-only tool executes immediately without pausing
print("1. Read-only Call:", gate.execute_or_pause("lookup_account", {"id": "ACC-101"}, []))

# 2. Irreversible tool triggers pause and persists state
pause_info = gate.execute_or_pause(
    "execute_refund", {"id": "ACC-101", "amount": 250.0}, history=[{"step": 1}]
)
print("\n2. Paused Call:", pause_info)

# 3. Resume with 'approve' -> executes the tool and updates history
run_id = pause_info["run_id"]
approved_result = gate.resume(run_id, decision="approve")
print("\n3. Resumed (Approved):", approved_result)

# 4. Another irreversible tool paused and then 'reject'ed
pause_info_2 = gate.execute_or_pause(
    "execute_refund", {"id": "ACC-999", "amount": 5000.0}, history=[]
)
rejected_result = gate.resume(pause_info_2["run_id"], decision="reject")
print("\n4. Resumed (Rejected):", rejected_result)

1. Read-only Call: {'status': 'executed', 'ok': True, 'result': 'CustomerAccount(id=ACC-101, balance=1250.00)'}

2. Paused Call: {'status': 'paused_for_approval', 'run_id': '1e65dc48-0cb8-4196-be3c-589897ab4f78', 'tool_name': 'execute_refund', 'arguments': {'id': 'ACC-101', 'amount': 250.0}, 'message': "Paused: 'execute_refund' is irreversible and requires human approval."}

3. Resumed (Approved): {'status': 'approved_and_executed', 'decision': 'approve', 'tool_name': 'execute_refund', 'payload': {'ok': True, 'result': 'Refunded $250.0 to account ACC-101'}, 'updated_history': [{'step': 1}, {'tool': 'execute_refund', 'payload': {'ok': True, 'result': 'Refunded $250.0 to account ACC-101'}}]}

4. Resumed (Rejected): {'status': 'rejected', 'decision': 'reject', 'tool_name': 'execute_refund', 'payload': {'ok': False, 'error': "Tool 'execute_refund' was rejected by human approver."}, 'updated_history': [{'tool': 'execute_refund', 'payload': {'ok': False, 'error': "Tool 'execute_refund' was r

### Task 7

In [41]:
@dataclass
class AgentStepRecord:
    step: int
    step_type: str  # 'llm' or 'tool'
    call: str | None = None  # tool name or prompt
    arguments: dict[str, Any] | None = None
    result: Any | None = None
    latency_ms: float = 0.0
    tokens: int = 0
    cost_usd: float = 0.0


@dataclass
class AgentRunRecord:
    run_id: str
    goal: str
    status: str
    steps: list[AgentStepRecord] = field(default_factory=list)
    total_latency_ms: float = 0.0
    total_tokens: int = 0
    total_cost_usd: float = 0.0


def serialize_agent_run(run: AgentRunRecord) -> str:
    """Serialize full agent run (steps, calls, results, latency, tokens, cost) to JSON."""
    return json.dumps(asdict(run), indent=2)


def replay_agent_run(json_data: str) -> AgentRunRecord:
    """Load serialized run JSON and replay the execution step-by-step."""
    data = json.loads(json_data)

    print(f"=== Replaying Run: {data['run_id']} ===")
    print(f"Goal: {data['goal']}")
    print(f"Status: {data['status']}\n")

    steps: list[AgentStepRecord] = []
    for s in data.get("steps", []):
        step_obj = AgentStepRecord(**s)
        steps.append(step_obj)

        if step_obj.step_type == "tool":
            print(
                f"[Step {step_obj.step}] TOOL CALL: {step_obj.call}({step_obj.arguments}) "
                f"-> Result: {step_obj.result} "
                f"| Latency: {step_obj.latency_ms:.1f}ms | Tokens: {step_obj.tokens} | Cost: ${step_obj.cost_usd:.6f}"
            )
        else:
            print(
                f"[Step {step_obj.step}] LLM REASONING: {step_obj.result} "
                f"| Latency: {step_obj.latency_ms:.1f}ms | Tokens: {step_obj.tokens} | Cost: ${step_obj.cost_usd:.6f}"
            )

    print("\n--- Replay Summary ---")
    print(f"Total Steps: {len(steps)}")
    print(f"Total Latency: {data['total_latency_ms']:.1f}ms")
    print(f"Total Tokens: {data['total_tokens']}")
    print(f"Total Cost: ${data['total_cost_usd']:.6f}")
    print("======================\n")

    return AgentRunRecord(
        run_id=data["run_id"],
        goal=data["goal"],
        status=data["status"],
        steps=steps,
        total_latency_ms=data["total_latency_ms"],
        total_tokens=data["total_tokens"],
        total_cost_usd=data["total_cost_usd"],
    )

### Task 8

In [45]:
class BudgetExceededError(RuntimeError):
    """Raised when accumulated run cost exceeds the maximum dollar budget."""


@dataclass(frozen=True)
class ModelPrice:
    prompt_per_1m: float
    completion_per_1m: float


# Default catalog (USD per 1M tokens)
DEFAULT_PRICING: dict[str, ModelPrice] = {
    "gemini-3.5-flash-lite": ModelPrice(prompt_per_1m=0.075, completion_per_1m=0.30),
    "gemini-3.8-flash": ModelPrice(prompt_per_1m=0.10, completion_per_1m=0.40),
    "gpt-4o-mini": ModelPrice(prompt_per_1m=0.15, completion_per_1m=0.60),
    "gpt-4o": ModelPrice(prompt_per_1m=2.50, completion_per_1m=10.00),
}


@dataclass
class CostTracker:
    """Converts per-step token usage into dollars and enforces a per-run cap."""

    max_cost_usd: float = 0.05
    pricing_table: dict[str, ModelPrice] = field(default_factory=lambda: dict(DEFAULT_PRICING))
    total_cost_usd: float = 0.0
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    steps_recorded: int = 0

    def calculate_step_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        """Calculate USD cost for a single turn using the pricing table."""
        price = self.pricing_table.get(
            model, ModelPrice(prompt_per_1m=0.075, completion_per_1m=0.30)
        )
        cost = (input_tokens / 1_000_000.0) * price.prompt_per_1m + (
            output_tokens / 1_000_000.0
        ) * price.completion_per_1m
        return round(cost, 8)

    def record_step(
        self,
        model: str,
        input_tokens: int,
        output_tokens: int,
    ) -> dict[str, float]:
        """Record step tokens, accumulate cost, and enforce per-run cap."""
        step_cost = self.calculate_step_cost(model, input_tokens, output_tokens)

        # Enforce per-run dollar budget cap
        if (self.total_cost_usd + step_cost) > self.max_cost_usd:
            raise BudgetExceededError(
                f"Per-run budget exceeded: Adding ${step_cost:.6f} to current "
                f"${self.total_cost_usd:.6f} exceeds max cap of ${self.max_cost_usd:.4f}."
            )

        self.total_cost_usd = round(self.total_cost_usd + step_cost, 8)
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.steps_recorded += 1

        return {
            "step_cost_usd": step_cost,
            "total_cost_usd": self.total_cost_usd,
            "remaining_budget_usd": round(max(0.0, self.max_cost_usd - self.total_cost_usd), 8),
        }

In [46]:
# Initialize tracker with a $0.001 per-run budget cap
tracker = CostTracker(max_cost_usd=0.001)

# Step 1: Initial query
s1 = tracker.record_step(model="gemini-3.5-flash-lite", input_tokens=1200, output_tokens=350)
print("Step 1:", s1)

# Step 2: Follow-up tool synthesis
s2 = tracker.record_step(model="gemini-3.5-flash-lite", input_tokens=1800, output_tokens=420)
print("Step 2:", s2)

print("\n--- Running Totals ---")
print(f"Steps Recorded: {tracker.steps_recorded}")
print(f"Total Input Tokens: {tracker.total_input_tokens}")
print(f"Total Output Tokens: {tracker.total_output_tokens}")
print(f"Total Cost: ${tracker.total_cost_usd:.6f}")

# Step 3: Enforcing budget cap
try:
    tracker.record_step(model="gemini-3.5-flash-lite", input_tokens=15_000_000, output_tokens=500)
except BudgetExceededError as exc:
    print("\nCap Enforced:", exc)

Step 1: {'step_cost_usd': 0.000195, 'total_cost_usd': 0.000195, 'remaining_budget_usd': 0.000805}
Step 2: {'step_cost_usd': 0.000261, 'total_cost_usd': 0.000456, 'remaining_budget_usd': 0.000544}

--- Running Totals ---
Steps Recorded: 2
Total Input Tokens: 3000
Total Output Tokens: 770
Total Cost: $0.000456

Cap Enforced: Per-run budget exceeded: Adding $1.125150 to current $0.000456 exceeds max cap of $0.0010.


### Task 9

implemented in test_harness.py

### Task 10

In [47]:
@dataclass
class EvalCase:
    goal: str
    expected_keyword: str | None = None


@dataclass
class EvalResult:
    goal: str
    success: bool
    steps: int
    cost_usd: float
    latency_ms: float
    output: str
    error: str | None = None


class EvaluationRunner:
    """Runs evaluation on N goals, collects telemetry metrics, and prints summary table."""

    def __init__(self, agent_fn: Callable[..., Any], cost_per_step: float = 0.00005) -> None:
        self.agent_fn = agent_fn
        self.cost_per_step = cost_per_step
        self.results: list[EvalResult] = []

    async def run(self, test_cases: list[EvalCase]) -> list[EvalResult]:
        """Execute evaluation across all test cases."""
        self.results = []
        for case in test_cases:
            t0 = time.perf_counter()
            try:
                output, steps = await self.agent_fn(case.goal)
                success = True
                if case.expected_keyword and case.expected_keyword.lower() not in output.lower():
                    success = False
                error = None
            except Exception as exc:  # noqa: BLE001
                output = ""
                steps = 1
                success = False
                error = str(exc)

            latency_ms = round((time.perf_counter() - t0) * 1000.0, 2)
            cost_usd = round(steps * self.cost_per_step, 6)

            self.results.append(
                EvalResult(
                    goal=case.goal,
                    success=success,
                    steps=steps,
                    cost_usd=cost_usd,
                    latency_ms=latency_ms,
                    output=output,
                    error=error,
                )
            )
        return self.results

    def print_summary(self) -> None:
        """Print a formatted summary table of all evaluated goals."""
        if not self.results:
            print("No evaluation results to display.")
            return

        print()
        print(
            f"{'#':<3} | {'Goal':<40} | {'Status':<8} | {'Steps':<6} | {'Cost ($)':<10} | {'Latency':<8}"
        )
        print("-" * 85)

        total_steps = 0
        total_cost = 0.0
        success_count = 0

        for idx, r in enumerate(self.results, 1):
            status = "PASS" if r.success else "FAIL"
            if r.success:
                success_count += 1
            total_steps += r.steps
            total_cost += r.cost_usd

            truncated_goal = r.goal if len(r.goal) <= 38 else r.goal[:35] + "..."
            print(
                f"{idx:<3} | {truncated_goal:<40} | {status:<8} | {r.steps:<6} | ${r.cost_usd:<9.6f} | {r.latency_ms:>6.1f}ms"
            )

        n = len(self.results)
        pass_rate = (success_count / n) * 100.0
        mean_steps = total_steps / n
        mean_cost = total_cost / n

        print("=" * 85)
        print(
            f"Summary: {n} Goals | Pass Rate: {pass_rate:.1f}% ({success_count}/{n}) | "
            f"Mean Steps: {mean_steps:.1f} | Total Cost: ${total_cost:.6f} | Mean Cost: ${mean_cost:.6f}"
        )
        print()

In [49]:
# Simulated mock agent function
async def mock_agent_eval(goal: str) -> tuple[str, int]:
    """Mock agent simulating execution: returns (output_text, steps_taken)."""
    await asyncio.sleep(0.01)
    if "fail" in goal.lower():
        raise RuntimeError("Service unavailable.")
    return f"Answer: Processed {goal}", 3


# 1. Define benchmark goals
benchmark_cases = [
    EvalCase(goal="Lookup current stock quote for AAPL", expected_keyword="AAPL"),
    EvalCase(goal="Convert $250 USD to EUR rate", expected_keyword="EUR"),
    EvalCase(goal="Check system metric memory usage", expected_keyword="Processed"),
    EvalCase(goal="Trigger a failing execution query", expected_keyword="Processed"),
]

# 2. Run evaluation suite
runner = EvaluationRunner(mock_agent_eval, cost_per_step=0.00004)
await runner.run(benchmark_cases)

# 3. Print evaluation summary table
runner.print_summary()


#   | Goal                                     | Status   | Steps  | Cost ($)   | Latency 
-------------------------------------------------------------------------------------
1   | Lookup current stock quote for AAPL      | PASS     | 3      | $0.000120  |   10.2ms
2   | Convert $250 USD to EUR rate             | PASS     | 3      | $0.000120  |   10.4ms
3   | Check system metric memory usage         | PASS     | 3      | $0.000120  |   10.2ms
4   | Trigger a failing execution query        | FAIL     | 1      | $0.000040  |   10.2ms
Summary: 4 Goals | Pass Rate: 75.0% (3/4) | Mean Steps: 2.5 | Total Cost: $0.000400 | Mean Cost: $0.000100

